# Module 8 Guided Lab: Data Integration with Pandas Merge

**BAN 6003: Data Management and Analytics Integration**

This week we focus on data integration using Pandas. In previous modules, we worked mostly within one table. Now we start combining related datasets.

The key idea for this module is:

> Data integration is not just putting tables together. It is making sure the combined result still represents what you think it represents.

We will use the `nycflights13` data because it includes multiple related tables: `flights`, `airlines`, `airports`, `planes`, and `weather`.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Explain keys, granularity, and relationships between datasets.
2. Identify primary key and foreign key candidates.
3. Use `pd.merge()` to combine tables.
4. Use `how="left"`, `how="inner"`, `how="outer"`, and `how="right"` appropriately.
5. Use `pd.concat()` to stack similar datasets.
6. Check row counts before and after merges.
7. Detect duplicate keys before merging.
8. Explain merge results in business language.

We will keep the code practical. We will not use complex loops or custom functions.

## Business Scenario

Imagine you are helping an airport analytics team prepare flight data for reporting.

The `flights` table tells us what happened for each flight. But it uses coded values such as carrier codes and airport codes.

For example, `carrier = "UA"` is not easy for a business audience unless we attach the airline name. `dest = "LAX"` is more useful if we also attach airport details. `tailnum` can connect a flight to aircraft details in the planes table.

This is why integration matters. One table rarely contains everything we need.

## 0. Setup: Load Packages and Tables

The repository requirements include `nycflights13`. In Codespaces they are installed during setup. When working locally, install `requirements.txt` before opening the notebook.

### Reproducibility Note

Keep package installation outside the analytical workflow. This notebook should focus on tables, keys, joins, and validation rather than changing the environment while it runs.

In [1]:
import nycflights13
import pandas as pd
from pathlib import Path

/opt/conda/lib/python3.12/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


In [2]:
flights = nycflights13.flights.copy()
airlines = nycflights13.airlines.copy()
airports = nycflights13.airports.copy()
planes = nycflights13.planes.copy()
weather = nycflights13.weather.copy()

Let's take a quick look at the tables we will use.

In [3]:
flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


In [4]:
airlines.head()

,carrier,name
0,9E,Endeavor Air Inc.
1,AA,American Airlines Inc.
2,AS,Alaska Airlines Inc.
3,B6,JetBlue Airways
4,DL,Delta Air Lines Inc.


In [5]:
airports.head()

,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


In [6]:
planes.head()

,tailnum,year,type,manufacturer,model,engines,seats,speed,engine
0,N10156,2004.0,Fixed wing multi engine,EMBRAER,EMB-145XR,2,55,NaN,Turbo-fan
1,N102UW,1998.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
2,N103US,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
3,N104UW,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
4,N10575,2002.0,Fixed wing multi engine,EMBRAER,EMB-145LR,2,55,NaN,Turbo-fan


## 1. Keys and Relationships

Before merging tables, we need to ask two questions:

1. What column connects the tables?
2. Is the key unique in the table we are merging from?

A **primary key** uniquely identifies rows in a table. A **foreign key** is a column in another table that refers to that primary key.

Example: `airlines["carrier"]` should identify one airline. `flights["carrier"]` refers to that airline code many times.

### Check key uniqueness in the lookup table

Before merging `flights` with `airlines`, check whether `carrier` is unique in the `airlines` table.

In [7]:
airlines["carrier"].duplicated().sum()

np.int64(0)

In [8]:
airlines["carrier"].nunique(), airlines.shape[0]

(16, 16)

If the number of unique carrier codes equals the number of rows, then `carrier` is unique in the `airlines` table.

Now check the `flights` table. We expect carrier codes to repeat there because there are many flights for each carrier.

In [9]:
flights["carrier"].duplicated().sum()

np.int64(336760)

In [10]:
flights["carrier"].value_counts().head()

carrier
UA    58665
B6    54635
EV    54173
DL    48110
AA    32729
Name: count, dtype: int64

This is normal. The same airline appears on many flights. That means the relationship is many flights to one airline.

### Your Turn 1

Check whether `faa` is unique in the `airports` table. Then check how often `dest` appears in the `flights` table.

In [11]:
airports["faa"].is_unique

True

In [12]:
flights["dest"].value_counts()

dest
ORD    17283
ATL    17215
LAX    16174
BOS    15508
MCO    14082
       ...  
MTJ       15
SBN       10
ANC        8
LEX        1
LGA        1
Name: count, Length: 105, dtype: int64

Write one sentence explaining the relationship between `flights["dest"]` and `airports["faa"]`.

**Your answer:**  
The values in airports["faa"] are unique airport codes and the values in flights["dest"] can appear multipe times since multiple flights can travel to the same destination airport. It shows many-to-one relationship where many rows in flights["dest"] match one row in airports["faa"].

## 2. Basic `pd.merge()` Pattern

The basic merge pattern is:

```python
pd.merge(left_table, right_table, how="left", left_on="column_in_left", right_on="column_in_right")
```

Important arguments:

- `left_table`: the main table you want to keep.
- `right_table`: the table you want to add information from.
- `how`: the type of join.
- `left_on`: the key column in the left table.
- `right_on`: the key column in the right table.

If the key column has the same name in both tables, use `on="key_column"`.

## 3. Left Join: Keep All Rows from the Main Table

### Business question

Can we attach airline names to every flight record?

We want to keep all flights and add the full airline name from the `airlines` table. This is a classic left join.

In [13]:
flights_with_airline = pd.merge(
    flights,
    airlines,
    how="left",
    on="carrier"
)

flights_with_airline[["carrier", "name", "flight", "origin", "dest"]].head()

,carrier,name,flight,origin,dest
0,UA,United Air Lines Inc.,1545,EWR,IAH
1,UA,United Air Lines Inc.,1714,LGA,IAH
2,AA,American Airlines Inc.,1141,JFK,MIA
3,B6,JetBlue Airways,725,JFK,BQN
4,DL,Delta Air Lines Inc.,461,LGA,ATL


### Check row counts after a left join

A left join should keep all rows from the left table. Here, the left table is `flights`. The row count should remain the same unless the right table has duplicate keys that create extra matches.

In [14]:
flights.shape[0], flights_with_airline.shape[0]

(336776, 336776)

In [15]:
flights_with_airline["name"].isna().sum()

np.int64(0)

The row count check tells us whether the merge unexpectedly created or lost rows. The missing value check tells us whether any flights had carrier codes that did not match the airline lookup table.

### Your Turn 2

Use a left join to attach destination airport information to the flights table.

Use:

- left table: `flights`
- right table: `airports`
- left key: `dest`
- right key: `faa`

Name the result `flights_with_dest_airport`.

In [16]:
flights_with_dest_airport = flights.merge(
    airports,
    how="left",
    left_on="dest",
    right_on="faa"
)

In [17]:
len(flights), len(flights_with_dest_airport)

(336776, 336776)

In [18]:
flights_with_dest_airport["name"].isna().sum()

np.int64(7602)

## 4. Inner Join: Keep Only Matching Rows

### Business question

What if we only want flight records that have matching airline information?

An inner join keeps only rows where the key appears in both tables.

In [19]:
flights_inner_airline = pd.merge(
    flights,
    airlines,
    how="inner",
    on="carrier"
)

flights.shape[0], flights_inner_airline.shape[0]

(336776, 336776)

In this case, the row count may be the same because all carriers in `flights` likely exist in `airlines`. But do not assume this will always happen. Inner joins can remove records. That can be correct, but it should be intentional.

### Example: inner join with planes

The `planes` table contains aircraft details by `tailnum`. Some flights may not have a matching `tailnum` in the `planes` table. Let's compare a left join and an inner join.

In [20]:
flights_planes_left = pd.merge(
    flights,
    planes,
    how="left",
    on="tailnum",
    suffixes=("_flight", "_plane")
)

flights_planes_inner = pd.merge(
    flights,
    planes,
    how="inner",
    on="tailnum",
    suffixes=("_flight", "_plane")
)

flights.shape[0], flights_planes_left.shape[0], flights_planes_inner.shape[0]

(336776, 336776, 284170)

In [21]:
flights_planes_left[["tailnum", "carrier", "flight", "manufacturer", "model", "seats"]].head()

,tailnum,carrier,flight,manufacturer,model,seats
0,N14228,UA,1545,BOEING,737-824,149.0
1,N24211,UA,1714,BOEING,737-824,149.0
2,N619AA,AA,1141,BOEING,757-223,178.0
3,N804JB,B6,725,AIRBUS,A320-232,200.0
4,N668DN,DL,461,BOEING,757-232,178.0


The inner join can reduce the dataset because it keeps only flights that have a matching aircraft record. In business work, this matters. If missing aircraft information is not random, an inner join may remove important records.

### Your Turn 3

Use an inner join to attach destination airport information to flights. Compare the row count with the original flights table. Did the inner join remove any rows?

In [22]:
flights_with_dest_airport_inner = flights.merge(
    airports,
    how="inner",
    left_on="dest",
    right_on="faa"
)

In [23]:
len(flights), len(flights_with_dest_airport_inner)

(336776, 329174)

**Your answer:**  
Did the inner join remove any rows? What does that suggest?
Yes the inner join removed 7,602 rows. This suggests that some destination airport codes in the flights table do not have matching airport records in the airports table. 

## 5. Outer Join: Keep Everything from Both Tables

### Business question

What if we want to audit whether there are airport codes that appear in one table but not the other?

An outer join keeps all rows from both tables. It is often useful for checking mismatches.

In [24]:
flight_dest_codes = flights[["dest"]].drop_duplicates().rename(columns={"dest": "faa"})

airport_audit = pd.merge(
    flight_dest_codes,
    airports[["faa", "name", "lat", "lon"]],
    how="outer",
    on="faa",
    indicator=True
)

airport_audit["_merge"].value_counts()

_merge
right_only    1357
both           101
left_only        4
Name: count, dtype: int64

In [25]:
airport_audit.head()

,faa,name,lat,lon,_merge
0,04G,Lansdowne Airport,41.130472,-80.619583,right_only
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,right_only
2,06C,Schaumburg Regional,41.989341,-88.101243,right_only
3,06N,Randall Airport,41.431912,-74.391561,right_only
4,09J,Jekyll Island Airport,31.074472,-81.427778,right_only


The `indicator=True` argument adds a special `_merge` column. This column tells us whether each row came from both tables, the left table only, or the right table only. This is useful for audit-style checks.

In [26]:
airport_audit.query("_merge != 'both'").head(10)

,faa,name,lat,lon,_merge
0,04G,Lansdowne Airport,41.130472,-80.619583,right_only
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,right_only
2,06C,Schaumburg Regional,41.989341,-88.101243,right_only
3,06N,Randall Airport,41.431912,-74.391561,right_only
4,09J,Jekyll Island Airport,31.074472,-81.427778,right_only
5,0A9,Elizabethton Municipal Airport,36.371222,-82.173417,right_only
6,0G6,Williams County Airport,41.467306,-84.506778,right_only
7,0G7,Finger Lakes Regional Airport,42.883565,-76.781232,right_only
8,0P2,Shoestring Aviation Airfield,39.794824,-76.647191,right_only
9,0S9,Jefferson County Intl,48.053809,-122.810644,right_only


### Your Turn 4

Create a list of unique origin airport codes from `flights`, rename the column to `faa`, and outer join it with `airports`. Use `indicator=True`. Then count the `_merge` categories.

In [27]:
origin_codes = flights[["origin"]].drop_duplicates().rename(
    columns={"origin": "faa"}
)

airport_audit = origin_codes.merge(
    airports,
    how="outer",
    on="faa",
    indicator=True
)

airport_audit["_merge"].value_counts()

_merge
right_only    1455
both             3
left_only        0
Name: count, dtype: int64

## 6. Right Join: Keep All Rows from the Right Table

Right joins are less common in Pandas workflows because we can usually switch the table order and use a left join. Still, it is helpful to understand what it does. A right join keeps all rows from the right table and matches rows from the left table when possible.

In [28]:
right_join_example = pd.merge(
    flights[["carrier", "flight", "origin", "dest"]].head(100),
    airlines,
    how="right",
    on="carrier"
)

right_join_example.head()

,carrier,flight,origin,dest,name
0,9E,NaN,NaN,NaN,Endeavor Air Inc.
1,AA,1141.0,JFK,MIA,American Airlines Inc.
2,AA,301.0,LGA,ORD,American Airlines Inc.
3,AA,707.0,LGA,DFW,American Airlines Inc.
4,AA,1895.0,EWR,MIA,American Airlines Inc.


In [29]:
right_join_example.shape

(105, 5)

In practice, I usually recommend using left joins when possible because they make the main table very clear. If your main table is `flights`, put `flights` on the left and use `how="left"`.

## 7. Many-to-Many Risk: Duplicate Keys Can Expand Rows

One common merge mistake is merging with a table that has duplicate keys. If the right table has multiple rows for the same key, one left row may match multiple right rows. This can expand the row count.

### A small example

Let's create a small fake lookup table with duplicate carrier keys.

In [30]:
fake_airline_regions = pd.DataFrame({
    "carrier": ["AA", "AA", "DL", "UA"],
    "region_label": ["legacy", "large", "legacy", "legacy"]
})

fake_airline_regions

,carrier,region_label
0,AA,legacy
1,AA,large
2,DL,legacy
3,UA,legacy


In [31]:
small_flights = flights[["carrier", "flight", "origin", "dest"]].query("carrier in ['AA', 'DL', 'UA']").head(10)

small_flights

,carrier,flight,origin,dest
0,UA,1545,EWR,IAH
1,UA,1714,LGA,IAH
2,AA,1141,JFK,MIA
4,DL,461,LGA,ATL
5,UA,1696,EWR,ORD
9,AA,301,LGA,ORD
12,UA,194,JFK,LAX
13,UA,1124,EWR,SFO
14,AA,707,LGA,DFW
16,UA,1187,EWR,LAS


In [32]:
merged_fake = pd.merge(
    small_flights,
    fake_airline_regions,
    how="left",
    on="carrier"
)

small_flights.shape[0], merged_fake.shape[0]

(10, 13)

In [33]:
merged_fake.head(10)

,carrier,flight,origin,dest,region_label
0,UA,1545,EWR,IAH,legacy
1,UA,1714,LGA,IAH,legacy
2,AA,1141,JFK,MIA,legacy
3,AA,1141,JFK,MIA,large
4,DL,461,LGA,ATL,legacy
5,UA,1696,EWR,ORD,legacy
6,AA,301,LGA,ORD,legacy
7,AA,301,LGA,ORD,large
8,UA,194,JFK,LAX,legacy
9,UA,1124,EWR,SFO,legacy


The row count increased because `AA` matched two rows in the fake lookup table. This is why we check duplicate keys before merging.

In [34]:
fake_airline_regions["carrier"].duplicated().sum()

np.int64(1)

### Your Turn 5

Check whether `tailnum` is unique in the `planes` table. Then explain whether merging `flights` with `planes` on `tailnum` is likely to create unexpected row expansion.

In [36]:
planes["tailnum"].is_unique

True

In [37]:
flights_planes = flights.merge(
    planes,
    how="left",
    on="tailnum"
)

len(flights), len(flights_planes)

(336776, 336776)

**Your answer:**  
The tailnum column is unique in the planes table which means each tail number appears only once. Merging flights with planes on tailnum does not create unexpected row expansion. The row count remained the same before and after the left join (336,776 rows) confirming that each flight matched at most one plane record.

## 8. `pd.concat()`: Stacking Similar Tables

`pd.merge()` combines tables side by side using keys.

`pd.concat()` stacks tables together when they have similar columns.

### Business question

Suppose we want one table containing flights from January and February. Each monthly table has the same columns, so we can stack them.

In [38]:
jan_flights = flights.query("month == 1")
feb_flights = flights.query("month == 2")

jan_flights.shape, feb_flights.shape

((27004, 19), (24951, 19))

In [39]:
jan_feb_flights = pd.concat([jan_flights, feb_flights], axis=0)

jan_feb_flights.shape

(51955, 19)

In [40]:
jan_feb_flights["month"].value_counts().sort_index()

month
1    27004
2    24951
Name: count, dtype: int64

Important arguments:

- `axis=0` stacks rows. This is the default.
- `axis=1` stacks columns side by side, but it should be used carefully because it aligns by index.
- `ignore_index=True` can reset the row index after stacking.

In [41]:
jan_feb_flights_clean_index = pd.concat(
    [jan_flights, feb_flights],
    axis=0,
    ignore_index=True
)

jan_feb_flights_clean_index.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


### Your Turn 6

Create a new DataFrame that stacks March and April flights. Then check the row count and month counts.

In [42]:
march_flights = flights[flights["month"] == 3]
april_flights = flights[flights["month"] == 4]

march_april_flights = pd.concat(
    [march_flights, april_flights],
    ignore_index=True
)

In [43]:
print("Row count:", len(march_april_flights))
print(march_april_flights["month"].value_counts())

Row count: 57164
month
3    28834
4    28330
Name: count, dtype: int64


## 9. Integration Checklist

Before and after integration, use this checklist:

1. What is the main table?
2. What is the lookup or secondary table?
3. What key connects the tables?
4. Is the key unique where it should be unique?
5. What type of join is appropriate?
6. Did the row count change as expected?
7. Did the merge create missing values?
8. Can you explain the result in business language?

## 10. Final Practice: Build an Integrated Flight Table

### Business Question

Create a flight-level table for July departures from JFK that is easier for business users to interpret. Start with a defined subset so the saved output remains compact.

Integrate:

- airline names from `airlines`; and
- destination airport names from `airports`.

Use left joins so every flight in the requested base table remains present. Name the final table `integrated_flights`.

In [44]:
# Final Practice Step 1: create the main flight-level table.
flight_base = flights.query("origin == 'JFK' & month == 7")[[
    "year", "month", "day", "carrier", "flight", "origin", "dest",
    "dep_delay", "arr_delay", "distance", "air_time"
]].copy()

flight_base.head()

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,distance,air_time
250450,2013,7,1,B6,915,JFK,SFO,212.0,157.0,2586,315.0
250451,2013,7,1,B6,1503,JFK,SJU,3.0,0.0,1598,200.0
250452,2013,7,1,B6,234,JFK,BTV,104.0,110.0,266,66.0
250454,2013,7,1,AA,185,JFK,LAX,174.0,120.0,2475,297.0
250455,2013,7,1,B6,165,JFK,PDX,235.0,186.0,2454,304.0


In [45]:
flights_with_names = flight_base.merge(
    airlines,
    how="left",
    on="carrier"
)

In [48]:
integrated_flights = flights_with_names.merge(
    airports[["faa", "name"]],
    how="left",
    left_on="dest",
    right_on="faa"
)


In [49]:
print("flight_base:", len(flight_base))
print("flights_with_names:", len(flights_with_names))
print("integrated_flights:", len(integrated_flights))

flight_base: 10023
flights_with_names: 10023
integrated_flights: 10023


In [50]:
print("Missing airline names:", flights_with_names["name"].isna().sum())
print("Missing destination airport names:", integrated_flights["name_y"].isna().sum())

Missing airline names: 0
Missing destination airport names: 589


In [51]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

if integrated_flights is None:
    print("Complete Final Practice Steps 2-3 before exporting.")
else:
    module8_output_path = output_dir / "integrated_jfk_july_flights_module8.csv"
    integrated_flights.to_csv(module8_output_path, index=False)
    print(f"Saved {len(integrated_flights):,} rows to: {module8_output_path}")


Saved 10,023 rows to: ../data/processed/integrated_jfk_july_flights_module8.csv


### Final Reflection

Write 4-6 sentences answering the following:

1. What was the main table and what does one row represent?
2. What lookup tables and keys did you use?
3. Why were left joins appropriate?
4. What row-count and missing-value evidence supports the result?
5. What business value did the added names provide?

**Your reflection:**  
The main table was flight_base, and each row represented a single flight departing from JFK in July 2013. I used the airlines table to add airline names using the carrier key and the airports table to add destination airport names using the dest and faa keys. Left joins were appropriate because they preserved every flight record from the main table while adding descriptive information when available. The row counts remained the same before and after the joins showing that no flight records were lost. Missing-value checks helped verify whether any airline or airport matches were unavailable. Adding airline and airport names makes the data easier for business users to read and analyze than relying on codes alone.

## 11. Offline Assignment - Aircraft Match Audit

The operations team asks whether aircraft manufacturer, model, and seat capacity can be added to `integrated_flights` without losing July JFK flights.

1. Check whether `tailnum` is unique in `planes`.
2. Add `tailnum` to the base flight selection if needed, then recreate the airline and destination joins.
3. Left join `manufacturer`, `model`, and `seats` from `planes` using `tailnum` and `indicator=True`.
4. Compare row counts, count merge statuses, and inspect missing aircraft fields.
5. Name the result `aircraft_match_audit`.

In [52]:
print("tailnum unique:", planes["tailnum"].is_unique)

flight_base = flights.query("origin == 'JFK' & month == 7")[
    [
        "year", "month", "day", "carrier", "flight",
        "origin", "dest", "tailnum",
        "dep_delay", "arr_delay", "distance", "air_time"
    ]
].copy()

flights_with_names = flight_base.merge(
    airlines,
    how="left",
    on="carrier"
)

integrated_flights = flights_with_names.merge(
    airports[["faa", "name"]],
    how="left",
    left_on="dest",
    right_on="faa"
)

aircraft_match_audit = integrated_flights.merge(
    planes[["tailnum", "manufacturer", "model", "seats"]],
    how="left",
    on="tailnum",
    indicator=True
)

print("\nRow Counts")
print("flight_base:", len(flight_base))
print("integrated_flights:", len(integrated_flights))
print("aircraft_match_audit:", len(aircraft_match_audit))

print("\nMerge Status Counts")
print(aircraft_match_audit["_merge"].value_counts())

print("\nMissing Aircraft Data")
print("manufacturer:", aircraft_match_audit["manufacturer"].isna().sum())
print("model:", aircraft_match_audit["model"].isna().sum())
print("seats:", aircraft_match_audit["seats"].isna().sum())

aircraft_match_audit.head()



tailnum unique: True

Row Counts
flight_base: 10023
integrated_flights: 10023
aircraft_match_audit: 10023

Merge Status Counts
_merge
both          8509
left_only     1514
right_only       0
Name: count, dtype: int64

Missing Aircraft Data
manufacturer: 1514
model: 1514
seats: 1514


,year,month,day,carrier,flight,origin,dest,tailnum,dep_delay,arr_delay,distance,air_time,name_x,faa,name_y,manufacturer,model,seats,_merge
0,2013,7,1,B6,915,JFK,SFO,N653JB,212.0,157.0,2586,315.0,JetBlue Airways,SFO,San Francisco Intl,AIRBUS,A320-232,200.0,both
1,2013,7,1,B6,1503,JFK,SJU,N805JB,3.0,0.0,1598,200.0,JetBlue Airways,NaN,NaN,AIRBUS,A320-232,200.0,both
2,2013,7,1,B6,234,JFK,BTV,N348JB,104.0,110.0,266,66.0,JetBlue Airways,BTV,Burlington Intl,EMBRAER,ERJ 190-100 IGW,20.0,both
3,2013,7,1,AA,185,JFK,LAX,N324AA,174.0,120.0,2475,297.0,American Airlines Inc.,LAX,Los Angeles Intl,BOEING,767-223,255.0,both
4,2013,7,1,B6,165,JFK,PDX,N640JB,235.0,186.0,2454,304.0,JetBlue Airways,PDX,Portland Intl,AIRBUS,A320-232,200.0,both


### Aircraft Integration Recommendation

**Your response here:** The left join preserved the flight-level unit because the row count remained the same before and after the aircraft merge. The tailnum key is unique in the planes table so the join did not create duplicate flight records or unexpected row expansion. The _merge counts provide evidence showing how many flights matched aircraft records and how many did not. An inner join could bias the analysis by removing flights with missing or unmatched aircraft information which would undercount actual flight activity. Unmatched aircraft records should be retained with null aircraft fields and flagged for follow-up data quality review.

## 12. Save Your Work

Before submitting:

1. Complete all Your Turn and final-practice cells.
2. Complete the aircraft match audit and both reflections.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm `data/processed/integrated_jfk_july_flights_module8.csv` exists.
5. Save the notebook, then commit and push your work.

Suggested commands:

```bash
git add .
git commit -m "Complete Module 8 pandas integration lab"
git push
```